# Raster aplicado · ¿Se puede medir la economía de un distrito desde el espacio?

En el Perú hay 1 873 distritos y **no existe PBI distrital**. El INEI lo publica a nivel
departamental, y con rezago. Entonces: ¿cómo estudiamos algo que varía entre distritos
—el efecto de una mina, de una carretera, de un programa social— si no tenemos la
variable dependiente?

Una respuesta que la literatura usa desde hace más de una década: **medir la luz que
emite cada lugar de noche**. Henderson, Storeygard y Weil (2012, *AER*) mostraron que la
luminosidad nocturna sigue de cerca al PBI, y hoy es un insumo estándar cuando los datos
oficiales faltan o llegan tarde.

En esta clase vamos a construir esa variable desde cero:

| Paso | Qué hacemos |
|---|---|
| 1 | Abrir el raster de luces nocturnas del Perú (VIIRS, 2021) |
| 2 | Pasar de píxeles a **tabla de distritos** — la operación que más van a usar |
| 3 | Validar que el resultado tiene sentido |
| 4 | Cruzarlo con un segundo raster: **áreas mineras** |
| 5 | Estimar si los distritos mineros concentran más actividad económica |

No hay teoría de sistemas de coordenadas ni catálogo de formatos. Cada concepto aparece
cuando hace falta para resolver el problema que tenemos delante.

## 0 · Preparación

Las dependencias están en el grupo `raster` del proyecto:

```bash
uv sync --group raster --group geo
```

In [ ]:
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

Los datos viven en Hugging Face, no en el repositorio. `hf_hub_download` los baja una vez
y los cachea: la segunda ejecución es instantánea.

In [ ]:
from huggingface_hub import hf_hub_download

REPO = "aquiro1994/ds-python-up"

def traer(ruta: str) -> str:
    """Descarga un archivo del dataset del curso y devuelve la ruta local."""
    return hf_hub_download(REPO, ruta, repo_type="dataset")

# Un shapefile no viaja solo: necesita sus archivos hermanos (.dbf, .shx, .prj).
# Los bajamos todos aunque solo abramos el .shp.
def traer_shapefile(base: str) -> str:
    principal = traer(f"{base}.shp")
    for ext in ("dbf", "shx", "prj", "cpg"):
        try:
            traer(f"{base}.{ext}")
        except Exception:
            pass          # .cpg y .prj son opcionales
    return principal

NTL_PERU = traer("_data/VIIRS_NTL_Peru_YearlyComposite_2021.tif")
print("descargado:", NTL_PERU.split("/")[-1])

## 1 · Qué es un raster, en 30 segundos

Un raster es **una matriz de números con coordenadas**. Nada más. Lo único que lo
distingue de un `np.array` cualquiera son dos piezas de metadata:

- **`transform`** — dónde cae el píxel (0,0) y cuánto mide cada celda
- **`crs`** — en qué sistema de coordenadas están esos números

Con eso ya se puede trabajar. Abramos el archivo.

In [ ]:
with rasterio.open(NTL_PERU) as src:
    luz = src.read(1)          # banda 1 -> np.ndarray 2D
    perfil = src.profile
    limites = src.bounds
    transform = src.transform
    crs = src.crs

print(f"tipo         : {type(luz).__name__}  {luz.shape}   ({luz.size:,} píxeles)")
print(f"dtype        : {luz.dtype}")
print(f"CRS          : {crs}")
print(f"resolución   : {src.res[0]:.5f}°  ≈ {src.res[0] * 111:.2f} km por píxel")
print(f"extensión    : lon [{limites.left:.2f}, {limites.right:.2f}]  "
      f"lat [{limites.bottom:.2f}, {limites.top:.2f}]")

Un raster de 1 412 × 2 039 píxeles cubriendo todo el Perú, a ~1 km por celda. Cada número
es radiancia: nanovatios por centímetro cuadrado por estereorradián. La unidad importa
poco; lo que importa es que **más luz = más actividad**.

## 2 · El primer problema real: los datos vienen sucios

Antes de graficar nada, mira los valores. Este paso se salta seguido y es donde se
esconden los errores que después no se explican.

In [ ]:
print(f"NaN            : {np.isnan(luz).sum():,}  ({np.isnan(luz).mean():.1%})")

validos = luz[~np.isnan(luz)]
print(f"píxeles válidos: {validos.size:,}")
print(f"  mínimo       : {validos.min():.3f}")
print(f"  mediana      : {np.median(validos):.3f}")
print(f"  percentil 99 : {np.percentile(validos, 99):.2f}")
print(f"  máximo       : {validos.max():.1f}")
print(f"  ≤ 0          : {(validos <= 0).mean():.1%}")

Dos hallazgos que cambian todo lo que viene después:

**El 27 % son `NaN`.** Es el rectángulo que cubre el archivo pero cae fuera del Perú —
mar, Brasil, Bolivia. Si calculas `luz.mean()` sin filtrar, obtienes `nan` y el resto del
análisis se cae en silencio.

**El 98 % de los píxeles válidos son cero o negativos.** El Perú de noche está casi
completamente oscuro. Los negativos no son un error: el sensor VIIRS resta un fondo
estimado y a veces la resta pasa de cero.

Y la distribución es brutalmente asimétrica: mediana 0, percentil 99 en 1.75, máximo en
323. Graficar eso en escala lineal no muestra nada.

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(11, 4.6))

izq.imshow(luz, cmap="inferno", vmin=0, vmax=validos.max())
izq.set_title("Escala lineal\n(un punto blanco = Lima; el resto, negro)", fontsize=9)
izq.axis("off")

luz_log = np.log1p(np.clip(luz, 0, None))
# el tope en el percentil 95 evita que Lima aplaste al resto del país
tope = np.nanpercentile(luz_log[luz_log > 0], 95)
der.imshow(luz_log, cmap="inferno", vmax=tope)
der.set_title("Escala logarítmica\nlog(1 + luz): aparece la red urbana", fontsize=9)
der.axis("off")

fig.suptitle("Luces nocturnas del Perú, 2021 — por qué la escala importa",
             fontsize=11, y=0.99)
plt.tight_layout()
plt.show()

El mismo dato, dos historias. En lineal solo se ve Lima porque concentra tanta luz que
aplasta al resto. En logarítmica aparecen la costa, la sierra y las ciudades amazónicas.

> **Regla práctica:** con luces nocturnas, población o ingresos, trabaja siempre en
> logaritmos. `log1p(x)` = `log(1+x)` porque el 98 % de los valores es cero y `log(0)`
> no existe.

## 3 · La operación que de verdad van a usar: raster → tabla

Un raster no sirve para regresiones. Lo que necesitamos es **una fila por distrito**.

Ese paso se llama *zonal statistics*: recortar el raster con cada polígono y resumir los
píxeles de adentro. Es, con diferencia, la operación más común al usar raster en
economía.

In [ ]:
DISTRITOS = traer_shapefile("_data/Folium/DISTRITOS")
distritos = gpd.read_file(DISTRITOS)

print(f"{len(distritos):,} distritos   CRS: EPSG:{distritos.crs.to_epsg()}")
distritos[["DEPARTAMEN", "PROVINCIA", "DISTRITO"]].head(3)

El CRS del shapefile (EPSG:4326) coincide con el del raster. **Si no coincidieran, el
recorte devolvería vacío o basura**, así que conviene comprobarlo siempre.

In [ ]:
assert distritos.crs.to_epsg() == crs.to_epsg(), "CRS distintos: reproyectar antes de cruzar"
print(f"raster y vector en el mismo CRS (EPSG:{crs.to_epsg()}) — se pueden cruzar")

In [ ]:
from rasterstats import zonal_stats

stats_luz = zonal_stats(
    distritos,                # los polígonos
    NTL_PERU,                 # el raster
    stats=["mean", "sum", "max", "count"],
    nodata=np.nan,            # explícito: si no, rasterstats inventa -999
    all_touched=True,         # incluye píxeles que el polígono toca, no solo los que cubre
)

luz_dist = pd.DataFrame(stats_luz)
print(f"{len(luz_dist)} filas — una por distrito")
luz_dist.head(3)

Tres argumentos que deciden si el resultado es correcto:

| Argumento | Por qué importa |
|---|---|
| `nodata=np.nan` | Sin esto, `rasterstats` avisa que asume `-999` y contamina los promedios |
| `all_touched=True` | Los distritos chicos de Lima miden menos que un píxel de 1 km: sin esto salen vacíos |
| `stats=[...]` | `sum` y `mean` responden preguntas distintas — lo vemos enseguida |

## 4 · Validar antes de creer

Ya tenemos un número por distrito. La pregunta obligatoria: **¿tiene sentido?**

Con datos peruanos la validación es barata, porque sabemos qué distritos deberían salir
arriba.

In [ ]:
panel = distritos[["DEPARTAMEN", "PROVINCIA", "DISTRITO", "geometry"]].copy()
panel["luz_media"] = luz_dist["mean"].values
panel["luz_total"] = luz_dist["sum"].values
panel["n_pixeles"] = luz_dist["count"].values

print("TOP 10 · luminosidad MEDIA (luz por km²)")
print(panel.nlargest(10, "luz_media")[["DISTRITO", "PROVINCIA", "luz_media"]]
      .to_string(index=False))

San Borja, Surquillo, Lince, La Victoria, San Isidro: el corazón comercial y financiero
de Lima. La medida está capturando algo real.

In [ ]:
print("TOP 10 · luminosidad TOTAL (luz acumulada del distrito)")
print(panel.nlargest(10, "luz_total")[["DISTRITO", "PROVINCIA", "luz_total"]]
      .to_string(index=False))

## 5 · La trampa: `mean` y `sum` responden preguntas distintas

Los dos rankings no coinciden, y la diferencia no es un detalle técnico.

- **`mean`** = luz por unidad de superficie → *intensidad*. Premia a los distritos
  chicos y densos (San Isidro).
- **`sum`** = luz acumulada → *escala*. Premia a los distritos grandes o muy poblados
  (San Juan de Lurigancho, Cusco).

Cuál usar depende de la pregunta. Para «¿qué tan urbano es este lugar?», `mean`. Para
«¿cuánta actividad económica hay en total?», `sum`. Elegir mal invierte el resultado.

In [ ]:
comparacion = pd.DataFrame({
    "por luz media": panel.nlargest(8, "luz_media")["DISTRITO"].values,
    "por luz total": panel.nlargest(8, "luz_total")["DISTRITO"].values,
})
comparacion.index = range(1, 9)
comparacion

## 6 · Segundo raster: dónde están las minas

Ahora la pregunta económica. Perú es un país minero, y una discusión vieja es si la
minería deja actividad económica en el territorio donde opera o si el valor se va a otra
parte.

Tenemos un raster global de áreas mineras (Maus et al., 2022) que registra km² de
superficie intervenida por celda. Le aplicamos exactamente el mismo procedimiento.

In [ ]:
MINAS = traer("_data/Global_mining/global_miningarea_v1_30arcsecond.tif")

with rasterio.open(MINAS) as src:
    print(f"raster global : {src.width:,} × {src.height:,} píxeles")
    print(f"resolución    : {src.res[0]:.5f}°  ≈ {src.res[0] * 111:.2f} km")
    print(f"nodata        : {src.nodata:.3g}")
    nodata_minas = src.nodata

Ojo con el `nodata`: es `-3.4e+38`, el mínimo de un `float32`. Si no se declara, entra al
promedio y produce números absurdos. **Siempre revisa el `nodata` antes de agregar.**

In [ ]:
stats_minas = zonal_stats(
    distritos, MINAS,
    stats=["sum"],
    nodata=nodata_minas,
    all_touched=True,
)

panel["mina_km2"] = pd.DataFrame(stats_minas)["sum"].fillna(0).values
panel["es_minero"] = (panel["mina_km2"] > 0).astype(int)

print(f"distritos con actividad minera detectada: {panel.es_minero.sum()} "
      f"de {len(panel)}  ({panel.es_minero.mean():.1%})")
print()
print(panel.nlargest(8, "mina_km2")[["DISTRITO", "DEPARTAMEN", "mina_km2"]]
      .to_string(index=False))

Esta es la mejor validación posible: el algoritmo no sabe nada de minería peruana y
recupera las operaciones más grandes del país.

| Distrito | Mina |
|---|---|
| Espinar (Cusco) | Antapaccay |
| Ilabaya (Tacna) | Toquepala |
| Torata (Moquegua) | Cuajone |
| Yarabamba (Arequipa) | Cerro Verde |
| San Marcos (Áncash) | Antamina |
| Marcona (Ica) | Shougang |
| Cajamarca | Yanacocha |
| Challhuahuacho (Apurímac) | Las Bambas |

Si el ranking hubiera devuelto distritos de Lima, sabríamos que algo está mal.

## 7 · Completar el panel

Falta el área de cada distrito. Y acá aparece el único momento de esta clase donde el
sistema de coordenadas es ineludible:

**EPSG:4326 mide en grados, no en metros.** Calcular áreas ahí devuelve «grados
cuadrados», que no significan nada. Hay que proyectar a un CRS métrico —para el Perú,
EPSG:24891.

In [ ]:
# mal: area en grados cuadrados
area_grados = distritos.geometry.area.iloc[0]

# bien: proyectar a un CRS en metros y pasar a km²
area_km2 = distritos.to_crs(epsg=24891).geometry.area.iloc[0] / 1e6

print(f"área del primer distrito")
print(f"  en EPSG:4326  -> {area_grados:.6f}   (grados² — no significa nada)")
print(f"  en EPSG:24891 -> {area_km2:,.1f} km²  (correcto)")

panel["area_km2"] = distritos.to_crs(epsg=24891).geometry.area.values / 1e6

In [ ]:
# tercer raster: precipitación anual (control climático)
PRECIP = traer("Labs/Python_Notebooks/LAB7/Prec_raster_peru.tif")

with rasterio.open(PRECIP) as src:
    nodata_precip = src.nodata

stats_precip = zonal_stats(distritos, PRECIP, stats=["mean"], band=1,
                           nodata=nodata_precip, all_touched=True)
panel["precip_mm"] = pd.DataFrame(stats_precip)["mean"].values

panel["log_luz"] = np.log1p(panel["luz_total"])
panel = panel.dropna(subset=["luz_total", "precip_mm"])

print(f"panel final: {panel.shape[0]:,} distritos × {panel.shape[1]} variables")
panel[["luz_total", "luz_media", "mina_km2", "area_km2", "precip_mm"]].describe().round(2)

## 8 · La pregunta: ¿los distritos mineros tienen más actividad económica?

Con el panel armado, la comparación descriptiva primero.

In [ ]:
resumen = (panel.groupby("es_minero")[["luz_total", "luz_media", "area_km2", "precip_mm"]]
           .agg(["mean", "median"]).round(2))
resumen.index = ["Sin minería", "Con minería"]
resumen

Un resultado que parece contradictorio y **no lo es**:

- Luz **total**: los distritos mineros tienen más (114 vs 89)
- Luz **media**: los distritos mineros tienen mucha menos (0.32 vs 1.54)

La explicación es geográfica. Los distritos mineros son grandes y mayormente vacíos —
puna, desierto, montaña—, así que su promedio por km² es bajo. Pero contienen un punto
muy brillante: el campamento, la planta, el tajo. La suma lo capta; el promedio lo diluye.

Esto conecta directo con la sección 5: **la elección entre `mean` y `sum` cambia el signo
de la conclusión**.

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(10.5, 4))

datos = [panel.loc[panel.es_minero == 0, "log_luz"],
         panel.loc[panel.es_minero == 1, "log_luz"]]
partes = izq.violinplot(datos, showmedians=True, widths=0.7)
for cuerpo, color in zip(partes["bodies"], ["#9aa7b0", "#c1663f"]):
    cuerpo.set_facecolor(color)
    cuerpo.set_alpha(0.75)
for pieza in ("cmedians", "cbars", "cmins", "cmaxes"):
    partes[pieza].set_color("#3c4a52")
izq.set_xticks([1, 2], ["Sin minería", "Con minería"])
izq.set_ylabel("log(1 + luz total)")
izq.set_title("Distribución de luminosidad", fontsize=9.5)

# los mineros son solo el 7 %: se dibujan encima para que no queden sepultados
sin = panel[panel.es_minero == 0]
con = panel[panel.es_minero == 1]
der.scatter(np.log(sin.area_km2), sin.log_luz, s=6, alpha=0.20,
            c="#9aa7b0", label="Sin minería")
der.scatter(np.log(con.area_km2), con.log_luz, s=16, alpha=0.85,
            c="#c1663f", edgecolors="white", linewidths=0.3, label="Con minería")
der.legend(frameon=False, fontsize=8, loc="upper left")
der.set_xlabel("log(área del distrito, km²)")
der.set_ylabel("log(1 + luz total)")
der.set_title("Luz vs tamaño del distrito", fontsize=9.5)

plt.tight_layout()
plt.show()

El gráfico de la derecha muestra por qué hace falta controlar: los distritos mineros
(naranja) están sistemáticamente a la derecha, o sea son **más grandes**. Un distrito
grande tiene más luz total solo por ser grande. Sin controlar el área, estaríamos midiendo
tamaño, no minería.

In [ ]:
import statsmodels.formula.api as smf

m1 = smf.ols("log_luz ~ es_minero", data=panel).fit()
m2 = smf.ols("log_luz ~ es_minero + np.log(area_km2)", data=panel).fit()
m3 = smf.ols("log_luz ~ es_minero + np.log(area_km2) + precip_mm + C(DEPARTAMEN)",
             data=panel).fit()

tabla = pd.DataFrame({
    "(1) simple": [m1.params["es_minero"], m1.bse["es_minero"], m1.rsquared, ""],
    "(2) + área": [m2.params["es_minero"], m2.bse["es_minero"], m2.rsquared, ""],
    "(3) + clima + depto.": [m3.params["es_minero"], m3.bse["es_minero"], m3.rsquared, "sí"],
}, index=["coef. minero", "error estándar", "R²", "efectos fijos depto."])

print(tabla.to_string())
print(f"\nN = {int(m3.nobs):,} distritos")
print(f"t del coeficiente en (3): {m3.tvalues['es_minero']:.2f}")

El coeficiente se mantiene alrededor de **1.3 log-puntos** en las tres especificaciones,
con un estadístico *t* superior a 8. Controlar por área, clima y departamento no lo mueve.

**Cómo leerlo:** un distrito con minería tiene aproximadamente
`exp(1.29) − 1 ≈ 2.6` veces más luminosidad total que uno comparable sin minería, dentro
del mismo departamento.

**Cómo NO leerlo:** esto *no* dice que la minería cause más actividad económica. Las minas
se ubican donde hay mineral, y esa ubicación puede correlacionar con carreteras, altitud o
historia de asentamiento. Es una correlación condicional bien medida —un buen punto de
partida—, no un efecto causal. Para eso harían falta variación temporal y un diseño de
identificación, que es materia de otro curso.

## 9 · El mapa del resultado

Un mapa coroplético cierra el análisis y sirve de última validación visual.

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(11, 7.5))

panel.plot(column="log_luz", cmap="inferno", ax=izq, linewidth=0,
           legend=True, legend_kwds={"label": "log(1 + luz total)", "shrink": 0.55})
izq.set_title("Actividad económica estimada\npor luminosidad nocturna", fontsize=10)
izq.axis("off")

panel.plot(color="#e8eaec", ax=der, linewidth=0)
panel[panel.es_minero == 1].plot(column="mina_km2", cmap="YlOrRd", ax=der,
                                 linewidth=0, legend=True,
                                 legend_kwds={"label": "km² de área minera", "shrink": 0.55})
der.set_title("Distritos con actividad minera\n(129 de 1 873)", fontsize=10)
der.axis("off")

plt.tight_layout()
plt.show()

El mapa de la izquierda reproduce la geografía económica conocida: la franja costera
iluminada, el eje Arequipa–Cusco–Puno, los enclaves amazónicos sobre los ríos. El de la
derecha muestra que la minería se concentra en la sierra sur y central.

## 10 · La resolución cambia lo que puedes ver

Un último punto práctico. Todo lo anterior usó píxeles de 1 km. Con un raster de Cusco a
~460 m, el mismo territorio revela detalle que antes se promediaba.

In [ ]:
CUSCO = traer("_data/Raster_2026/VNL_cusco_2025.tif")

with rasterio.open(CUSCO) as src:
    luz_cusco = src.read(1)
    print(f"Cusco 2025: {src.width} × {src.height} px | "
          f"{src.res[0] * 111 * 1000:.0f} m por píxel")

fig, ax = plt.subplots(figsize=(7, 6.4))
ax.imshow(np.log1p(np.clip(luz_cusco, 0, None)), cmap="inferno")
ax.set_title("Cusco de noche, 2025 · ~460 m por píxel\n"
             "se distinguen la ciudad, el Valle Sagrado y las vías", fontsize=9.5)
ax.axis("off")
plt.tight_layout()
plt.show()

**El compromiso:** más resolución permite estudiar unidades más chicas (centros poblados,
manzanas), pero multiplica el peso del archivo y el tiempo de cómputo. Para trabajar a
nivel distrital, 1 km sobra. Para estudiar un corredor vial o un centro poblado, no
alcanza.

## Lo que aprendieron

El flujo completo, que se repite casi igual para cualquier raster:

```python
raster  = rasterio.open(archivo)          # 1. abrir
valores = raster.read(1)                  # 2. inspeccionar: NaN, nodata, escala
stats   = zonal_stats(polígonos, archivo, # 3. agregar a unidades administrativas
                      stats=["mean", "sum"],
                      nodata=<el correcto>,
                      all_touched=True)
panel   = pd.DataFrame(stats)             # 4. panel -> pandas -> regresión
```

Los cuatro errores que más cuestan:

1. **No declarar `nodata`** — el relleno entra al promedio y arruina el resultado
2. **Ignorar los `NaN`** — `mean()` devuelve `nan` y el análisis falla en silencio
3. **Calcular áreas o distancias en EPSG:4326** — grados no son metros
4. **Confundir `sum` con `mean`** — responden preguntas distintas y pueden dar signos opuestos

## Ejercicios

**1 · Precipitación y luz.** El raster de precipitación tiene 5 bandas (5 períodos).
Extrae la banda 3 y estima si los distritos más lluviosos son más o menos luminosos.
Controla por altitud usando el área como proxy imperfecto y comenta la limitación.

**2 · El umbral importa.** Definimos «minero» como `mina_km2 > 0`, incluyendo distritos
con apenas 0.01 km². Repite la regresión con umbrales de 1, 5 y 10 km². ¿El coeficiente
crece, se mantiene o desaparece? ¿Qué dice eso sobre el resultado?

**3 · Intensidad vs escala.** Corre la regresión principal usando `luz_media` en lugar de
`luz_total`. El signo se invierte. Explica por qué, y argumenta cuál de las dos
especificaciones responde mejor a la pregunta «¿la minería deja actividad económica en el
territorio?».

**4 · Tu propio distrito.** Elige un distrito que conozcas. Extrae su serie de luminosidad,
compárala con la mediana de su provincia y contrasta el resultado con lo que sabes del
lugar. ¿La medida captura bien lo que ves?